<a href="https://colab.research.google.com/github/MukeshMoharana/Head-and-Neck-Tumor-Segmentation-Using-HITL_MIRS-NET/blob/main/MIRS_Net_Stage3_Memory_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 45.3 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import nibabel as nib

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from monai.networks.nets import SwinUNETR

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [4]:
PROJECT_DIR = "/content/drive/MyDrive/MIRS-Net"

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR,
    "checkpoints"
)

BEST_STAGE2_MODEL = os.path.join(
    CHECKPOINT_DIR,
    "best_stage2_model.pth"
)

METADATA_PATH = os.path.join(
    PROJECT_DIR,
    "metadata.csv"
)

In [5]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

stage2_model = SwinUNETR(
    in_channels=4,
    out_channels=2,
    feature_size=48,
    patch_size=2,
    use_checkpoint=True
)

stage2_model.load_state_dict(
    torch.load(
        BEST_STAGE2_MODEL,
        map_location=device
    )
)

stage2_model.eval()

print("Stage 2 checkpoint loaded successfully")

Stage 2 checkpoint loaded successfully


In [6]:
# print(stage2_model)

# for name, module in stage2_model.named_children():
#     print(name)

In [7]:
# x = torch.randn(1,4,96,96,96)

# with torch.no_grad():

#     y = stage2_model.swinViT(x)

# print(type(y))

# if isinstance(y, (list, tuple)):
#     print(len(y))

#     for i, feat in enumerate(y):
#         print(i, feat.shape)

In [8]:
class Stage3Backbone(nn.Module):

    def __init__(self, stage2_model):

        super().__init__()

        self.stage2_model = stage2_model

    def forward(self, x):

        swin_features = (
            self.stage2_model.swinViT(x)
        )

        logits = self.stage2_model(x)

        return {
            "logits": logits,
            "features": swin_features,
            "bottleneck": swin_features[-1]
        }

In [9]:
backbone = Stage3Backbone(
    stage2_model
)

# x = torch.randn(
#     1,
#     4,
#     96,
#     96,
#     96
# )

# with torch.no_grad():

#     output = backbone(x)
#     logits = output["logits"]
#     features = output["features"]
#     bottleneck = output["bottleneck"]

# print("Logits:", logits.shape)
# print("Bottleneck:", bottleneck.shape)

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class AppearancePrototypeEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.pool = nn.AdaptiveAvgPool3d(1)

        self.proj = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 256)
        )

    def forward(self, bottleneck):

        x = self.pool(bottleneck)

        x = x.flatten(1)

        x = self.proj(x)

        x = F.normalize(x, p=2, dim=1)

        return x

In [11]:
appearance_encoder = AppearancePrototypeEncoder()

# dummy_bottleneck = torch.randn(
#     1,
#     768,
#     3,
#     3,
#     3
# )

# appearance_proto = appearance_encoder(
#     dummy_bottleneck
# )

# print(
#     appearance_proto.shape
# )

In [12]:
class ShapeEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv3d(1, 32, 3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),

            nn.MaxPool3d(2),

            nn.Conv3d(32, 64, 3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool3d(2),

            nn.Conv3d(64, 128, 3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True)
        )

        self.pool = nn.AdaptiveAvgPool3d(1)

    def forward(self, mask):

        x = self.encoder(mask)

        x = self.pool(x)

        x = x.flatten(1)

        x = F.normalize(x, p=2, dim=1)

        return x

In [13]:
shape_encoder = ShapeEncoder()

# dummy_mask = torch.randint(
#     0,
#     2,
#     (1,1,96,96,96)
# ).float()

# shape_proto = shape_encoder(
#     dummy_mask
# )

# print(shape_proto.shape)

In [14]:
import numpy as np
from scipy.ndimage import binary_dilation
from scipy.ndimage import binary_erosion


def create_boundary_mask(mask_np):

    dilated = binary_dilation(mask_np)

    eroded = binary_erosion(mask_np)

    boundary = dilated.astype(np.float32) - eroded.astype(np.float32)

    return boundary

In [15]:
class BoundaryEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv3d(1, 32, 3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),

            nn.MaxPool3d(2),

            nn.Conv3d(32, 64, 3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),

            nn.MaxPool3d(2),

            nn.Conv3d(64, 128, 3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True)
        )

        self.pool = nn.AdaptiveAvgPool3d(1)

    def forward(self, boundary):

        x = self.encoder(boundary)

        x = self.pool(x)

        x = x.flatten(1)

        x = F.normalize(x, p=2, dim=1)

        return x

In [16]:
boundary_encoder = BoundaryEncoder()

# dummy_boundary = torch.randint(
#     0,
#     2,
#     (1,1,96,96,96)
# ).float()

# boundary_proto = boundary_encoder(
#     dummy_boundary
# )

# print(boundary_proto.shape)

In [17]:
# memory_key = torch.cat(
#     [
#         appearance_proto,
#         shape_proto,
#         boundary_proto
#     ],
#     dim=1
# )
# print(memory_key.shape)

In [18]:
def create_boundary_mask(mask_tensor):

    boundary_batch = []

    mask_np = mask_tensor.detach().cpu().numpy()

    for b in range(mask_np.shape[0]):

        current_mask = mask_np[b, 0]

        current_mask = (
            current_mask > 0.5
        ).astype(np.uint8)

        dilated = binary_dilation(
            current_mask
        )

        eroded = binary_erosion(
            current_mask
        )

        boundary = (
            dilated.astype(np.float32)
            -
            eroded.astype(np.float32)
        )

        boundary_batch.append(
            boundary
        )

    boundary_batch = np.stack(
        boundary_batch,
        axis=0
    )

    boundary_batch = torch.tensor(
        boundary_batch,
        dtype=torch.float32
    )

    boundary_batch = (
        boundary_batch.unsqueeze(1)
    )

    return boundary_batch

In [19]:
class PrototypeGenerator(nn.Module):

    def __init__(
        self,
        appearance_encoder,
        shape_encoder,
        boundary_encoder
    ):

        super().__init__()

        self.appearance_encoder = appearance_encoder

        self.shape_encoder = shape_encoder

        self.boundary_encoder = boundary_encoder

    def forward(
        self,
        bottleneck,
        mask
    ):

        appearance_proto = (
            self.appearance_encoder(
                bottleneck
            )
        )

        shape_proto = (
            self.shape_encoder(
                mask
            )
        )

        boundary_mask = (
            create_boundary_mask(mask)
        )

        boundary_mask = (
            boundary_mask.to(mask.device)
        )

        boundary_proto = (
            self.boundary_encoder(
                boundary_mask
            )
        )

        memory_key = torch.cat(
            [
                appearance_proto,
                shape_proto,
                boundary_proto
            ],
            dim=1
        )

        return {
            "appearance": appearance_proto,
            "shape": shape_proto,
            "boundary": boundary_proto,
            "key": memory_key
        }

In [20]:
prototype_generator = PrototypeGenerator(
    appearance_encoder,
    shape_encoder,
    boundary_encoder
)

In [21]:
# dummy_bottleneck = torch.randn(
#     1,
#     768,
#     3,
#     3,
#     3
# )

# dummy_mask = torch.randint(
#     0,
#     2,
#     (1,1,96,96,96)
# ).float()

# outputs = prototype_generator(
#     dummy_bottleneck,
#     dummy_mask
# )

# for k, v in outputs.items():

#     print(
#         k,
#         v.shape
#     )

In [22]:
def generate_2d_bbox(
        mask,
        min_margin=5,
        max_margin=20):

    # Find slices containing tumor
    tumor_slices = np.where(
        np.sum(mask > 0, axis=(0,1)) > 0
    )[0]

    # Random tumor slice
    selected_slice = np.random.choice(
        tumor_slices
    )

    mask_slice = mask[:,:,selected_slice]

    coords = np.argwhere(
        mask_slice > 0
    )

    ymin, xmin = coords.min(axis=0)
    ymax, xmax = coords.max(axis=0)

    margin = np.random.randint(
        min_margin,
        max_margin + 1
    )

    xmin = max(0, xmin - margin)
    ymin = max(0, ymin - margin)

    xmax = min(
        mask.shape[0]-1,
        xmax + margin
    )

    ymax = min(
        mask.shape[1]-1,
        ymax + margin
    )

    return {
        "slice_idx": selected_slice,
        "bbox": [
            xmin,
            ymin,
            xmax,
            ymax
        ]
    }

In [23]:
def generate_memory_bbox(mask):

    tumor_slices = np.where(
        np.sum(mask > 0, axis=(0,1)) > 0
    )[0]

    selected_slice = tumor_slices[
        len(tumor_slices) // 2
    ]

    mask_slice = mask[:, :, selected_slice]

    coords = np.argwhere(mask_slice > 0)

    ymin, xmin = coords.min(axis=0)
    ymax, xmax = coords.max(axis=0)

    margin = 10

    xmin = max(0, xmin - margin)
    ymin = max(0, ymin - margin)

    xmax = min(
        mask.shape[0]-1,
        xmax + margin
    )

    ymax = min(
        mask.shape[1]-1,
        ymax + margin
    )

    return {
        "slice_idx": selected_slice,
        "bbox": [
            xmin,
            ymin,
            xmax,
            ymax
        ]
    }

In [24]:
def create_prompt_volume(
        shape,
        slice_idx,
        bbox,
        radius=3):

    prompt = np.zeros(
        shape,
        dtype=np.float32
    )

    xmin,ymin,xmax,ymax = bbox

    start_slice = max(
        0,
        slice_idx - radius
    )

    end_slice = min(
        shape[2]-1,
        slice_idx + radius
    )

    for z in range(
        start_slice,
        end_slice + 1
    ):

        prompt[
            xmin:xmax,
            ymin:ymax,
            z
        ] = 1.0

    return prompt

In [25]:
import numpy as np
from scipy.ndimage import binary_dilation


def create_negative_click_center(mask):

    boundary_region = (
        binary_dilation(
            mask > 0,
            iterations=5
        )
        &
        ~(mask > 0)
    )

    candidates = np.argwhere(
        boundary_region
    )

    idx = np.random.randint(
        len(candidates)
    )

    return candidates[idx]

In [26]:
def create_gaussian_click_map(
        shape,
        center,
        sigma=5):

    x, y, z = center

    xx, yy, zz = np.ogrid[
        :shape[0],
        :shape[1],
        :shape[2]
    ]

    distance_sq = (
        (xx - x) ** 2 +
        (yy - y) ** 2 +
        (zz - z) ** 2
    )

    gaussian = np.exp(
        -distance_sq /
        (2 * sigma**2)
    )

    return gaussian.astype(
        np.float32
    )

In [27]:
from scipy.ndimage import binary_erosion

def create_positive_click_volume(
        mask,
        sigma=5):

    tumor_core = binary_erosion(
        mask > 0,
        iterations=3
    )

    candidates = np.argwhere(
        tumor_core
    )

    if len(candidates) == 0:

        candidates = np.argwhere(
            mask > 0
        )

    center = candidates[
        np.random.randint(
            len(candidates)
        )
    ]

    return create_gaussian_click_map(
        mask.shape,
        center,
        sigma=sigma
    )

In [28]:
def create_memory_positive_click_volume(
    mask,
    sigma=5
):

    tumor_voxels = np.argwhere(
        mask > 0
    )

    center = (
        tumor_voxels.mean(axis=0)
    ).astype(int)

    return create_gaussian_click_map(
        mask.shape,
        center,
        sigma=sigma
    )

In [29]:
def create_negative_click_volume(
        mask,
        sigma=5):

    center = create_negative_click_center(
        mask
    )

    click = create_gaussian_click_map(
        mask.shape,
        center,
        sigma=sigma
    )

    click[mask > 0] = 0

    return click

In [30]:
from scipy.spatial.distance import cdist

def create_memory_negative_click_volume(
    mask,
    sigma=5
):

    tumor_voxels = np.argwhere(
        mask > 0
    )

    centroid = (
        tumor_voxels.mean(axis=0)
    )

    boundary_region = (
        binary_dilation(
            mask > 0,
            iterations=5
        )
        &
        ~(mask > 0)
    )

    candidates = np.argwhere(
        boundary_region
    )

    distances = cdist(
        candidates,
        centroid.reshape(1,-1)
    )

    center = candidates[
        np.argmin(distances)
    ]

    click = create_gaussian_click_map(
        mask.shape,
        center,
        sigma=sigma
    )

    click[mask > 0] = 0

    return click

In [31]:
def extract_foreground_patch(
    image,
    mask,
    prompt,
    positive_click,
    negative_click,
    patch_size=(96,96,96)
):
    tumor_voxels = np.argwhere(mask > 0)

    if len(tumor_voxels) == 0:
        raise ValueError("Empty mask found")

    center_idx = np.random.randint(len(tumor_voxels))
    cz, cy, cx = tumor_voxels[center_idx]

    hz, hy, hx = [p // 2 for p in patch_size]

    image = np.pad(
        image,
        ((hz,hz),(hy,hy),(hx,hx)),
        mode="constant"
    )

    mask = np.pad(
        mask,
        ((hz,hz),(hy,hy),(hx,hx)),
        mode="constant"
    )

    prompt = np.pad(
        prompt,
        ((hz,hz),(hy,hy),(hx,hx)),
        mode="constant"
    )
    positive_click = np.pad(
      positive_click,
      ((hz,hz),(hy,hy),(hx,hx)),
      mode="constant"
    )

    negative_click = np.pad(
      negative_click,
      ((hz,hz),(hy,hy),(hx,hx)),
      mode="constant"
    )

    cz += hz
    cy += hy
    cx += hx

    image_patch = image[
        cz-hz:cz+hz,
        cy-hy:cy+hy,
        cx-hx:cx+hx
    ]

    mask_patch = mask[
        cz-hz:cz+hz,
        cy-hy:cy+hy,
        cx-hx:cx+hx
    ]

    prompt_patch = prompt[
        cz-hz:cz+hz,
        cy-hy:cy+hy,
        cx-hx:cx+hx
    ]
    positive_click_patch = positive_click[
      cz-hz:cz+hz,
      cy-hy:cy+hy,
      cx-hx:cx+hx
    ]

    negative_click_patch = negative_click[
      cz-hz:cz+hz,
      cy-hy:cy+hy,
      cx-hx:cx+hx
    ]

    return (
    image_patch,
    mask_patch,
    prompt_patch,
    positive_click_patch,
    negative_click_patch
    )

In [32]:
def extract_center_patch(
    image,
    mask,
    prompt,
    positive_click,
    negative_click,
    patch_size=(96,96,96)
):

    tumor_voxels = np.argwhere(mask > 0)

    center = tumor_voxels.mean(
        axis=0
    ).astype(int)

    dz, dy, dx = patch_size

    z, y, x = center

    z1 = max(0, z - dz // 2)
    y1 = max(0, y - dy // 2)
    x1 = max(0, x - dx // 2)

    z2 = z1 + dz
    y2 = y1 + dy
    x2 = x1 + dx

    if z2 > image.shape[0]:
        z2 = image.shape[0]
        z1 = z2 - dz

    if y2 > image.shape[1]:
        y2 = image.shape[1]
        y1 = y2 - dy

    if x2 > image.shape[2]:
        x2 = image.shape[2]
        x1 = x2 - dx

    image = image[z1:z2, y1:y2, x1:x2]
    mask = mask[z1:z2, y1:y2, x1:x2]
    prompt = prompt[z1:z2, y1:y2, x1:x2]

    positive_click = positive_click[
        z1:z2,
        y1:y2,
        x1:x2
    ]

    negative_click = negative_click[
        z1:z2,
        y1:y2,
        x1:x2
    ]

    return (
        image,
        mask,
        prompt,
        positive_click,
        negative_click
    )

In [33]:
import torch
from torch.utils.data import Dataset
class HNCDataset(Dataset):

    def __init__(self,
                 metadata_csv,
                 split):

        self.df = pd.read_csv(metadata_csv)

        self.df = self.df[
            self.df["split"] == split
        ].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        try:
            row = self.df.iloc[idx]

            image = nib.load(
                row["image_path"]
            ).get_fdata()

            mask = nib.load(
                row["mask_path"]
            ).get_fdata()
            case_id = row["case_id"]
            # -------------------------
            # intensity normalization
            # -------------------------

            image = image.astype(np.float32)

            image = (
                image - image.mean()
            ) / (
                image.std() + 1e-8
            )

            # -------------------------
            # prompt generation
            # -------------------------

            bbox_info = generate_2d_bbox(mask)

            prompt = create_prompt_volume(
                image.shape,
                bbox_info["slice_idx"],
                bbox_info["bbox"],
                radius=3
            )

            # positive and negaitive click
            positive_click = create_positive_click_volume(
              mask)

            negative_click = create_negative_click_volume(
              mask)

            # -------------------------
            # convert to channels-first
            # -------------------------

            image = np.transpose(
                image,
                (2, 0, 1)
            )

            prompt = np.transpose(
                prompt,
                (2, 0, 1)
            )

            mask = np.transpose(
                mask,
                (2, 0, 1)
            )
            positive_click = np.transpose(
              positive_click,
              (2,0,1))

            negative_click = np.transpose(
              negative_click,
              (2,0,1))

            image,mask,prompt,positive_click,negative_click = extract_foreground_patch(
              image,
              mask,
              prompt,
              positive_click,
              negative_click,
              patch_size=(96,96,96))


            input_tensor = np.stack(
                [image, prompt,positive_click,negative_click],
                axis=0
            )


            return {
                "case_id": case_id,
                "image": torch.tensor(
                    input_tensor,
                    dtype=torch.float32
                ),

                "mask": torch.tensor(
                    mask,
                    dtype=torch.float32
                )
            }
        except Exception as e:
            print(f"Error processing index {idx}: {e}")
            print(f"Problematic row data: {self.df.iloc[idx]}")
            raise # Re-raise the exception after printing for full traceback

In [34]:
# df = pd.read_csv(METADATA_PATH)

# print(df.columns)

In [35]:
# import torch
# from torch.utils.data import Dataset
# class MemoryDataset(Dataset):

#     def __init__(self,
#                  metadata_csv,
#                  split):

#         self.df = pd.read_csv(metadata_csv)

#         self.df = self.df[
#             self.df["split"] == split
#         ].reset_index(drop=True)

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         try:
#             row = self.df.iloc[idx]

#             image = nib.load(
#                 row["image_path"]
#             ).get_fdata()

#             mask = nib.load(
#                 row["mask_path"]
#             ).get_fdata()
#             case_id = row["case_id"]
#             # -------------------------
#             # intensity normalization
#             # -------------------------

#             image = image.astype(np.float32)

#             image = (
#                 image - image.mean()
#             ) / (
#                 image.std() + 1e-8
#             )

#             # -------------------------
#             # prompt generation
#             # -------------------------

#             bbox_info = generate_memory_bbox(mask)

#             prompt = create_prompt_volume(
#                 image.shape,
#                 bbox_info["slice_idx"],
#                 bbox_info["bbox"],
#                 radius=3
#             )

#             # positive and negaitive click
#             positive_click = create_memory_positive_click_volume(
#               mask)

#             negative_click = create_memory_negative_click_volume(
#               mask)

#             # -------------------------
#             # convert to channels-first
#             # -------------------------

#             image = np.transpose(
#                 image,
#                 (2, 0, 1)
#             )

#             prompt = np.transpose(
#                 prompt,
#                 (2, 0, 1)
#             )

#             mask = np.transpose(
#                 mask,
#                 (2, 0, 1)
#             )
#             positive_click = np.transpose(
#               positive_click,
#               (2,0,1))

#             negative_click = np.transpose(
#               negative_click,
#               (2,0,1))

#             image,mask,prompt,positive_click,negative_click = extract_center_patch(
#               image,
#               mask,
#               prompt,
#               positive_click,
#               negative_click,
#               patch_size=(96,96,96))


#             input_tensor = np.stack(
#                 [image, prompt,positive_click,negative_click],
#                 axis=0
#             )


#             return {
#                 "case_id": case_id,
#                 "image": torch.tensor(
#                     input_tensor,
#                     dtype=torch.float32
#                 ),

#                 "mask": torch.tensor(
#                     mask,
#                     dtype=torch.float32
#                 )
#             }
#         except Exception as e:
#             print(f"Error processing index {idx}: {e}")
#             print(f"Problematic row data: {self.df.iloc[idx]}")
#             raise # Re-raise the exception after printing for full traceback

In [36]:
# memory_dataset = MemoryDataset(
#     METADATA_PATH,
#     "train"
# )

# sample = memory_dataset[0]

# print(sample["case_id"])
# print(sample["image"].shape)
# print(sample["mask"].shape)

In [37]:
stage2_model.eval()

appearance_encoder.eval()

shape_encoder.eval()

boundary_encoder.eval()

prototype_generator.eval()

PrototypeGenerator(
  (appearance_encoder): AppearancePrototypeEncoder(
    (pool): AdaptiveAvgPool3d(output_size=1)
    (proj): Sequential(
      (0): Linear(in_features=768, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=256, bias=True)
    )
  )
  (shape_encoder): ShapeEncoder(
    (encoder): Sequential(
      (0): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (5): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU(inplace=True)
      (7): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (8): Conv3d(64, 128, kernel_size=(3, 3, 

In [38]:
stage2_model.eval()
prototype_generator.eval()

# loader = DataLoader(
#     memory_dataset,
#     batch_size=1,
#     shuffle=False
# )

# sample = next(iter(loader))

# image = sample["image"]
# mask = sample["mask"].unsqueeze(1)

# with torch.no_grad():

#     backbone_output = backbone(image)
#     logits = backbone_output["logits"]
#     bottleneck = backbone_output["bottleneck"]

#     outputs = prototype_generator(
#         bottleneck,
#         mask
#     )

# for k, v in outputs.items():

#     print(k, v.shape)

PrototypeGenerator(
  (appearance_encoder): AppearancePrototypeEncoder(
    (pool): AdaptiveAvgPool3d(output_size=1)
    (proj): Sequential(
      (0): Linear(in_features=768, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=256, bias=True)
    )
  )
  (shape_encoder): ShapeEncoder(
    (encoder): Sequential(
      (0): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (5): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU(inplace=True)
      (7): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (8): Conv3d(64, 128, kernel_size=(3, 3, 

In [39]:
# class MemoryBankBuilder:

#     def __init__(
#         self,
#         backbone,
#         prototype_generator,
#         device
#     ):

#         self.backbone = backbone.to(device)

#         self.prototype_generator = (
#             prototype_generator.to(device)
#         )

#         self.device = device

#         self.backbone.eval()
#         self.prototype_generator.eval()

#     @torch.no_grad()
#     def build_memory_bank(
#         self,
#         dataloader
#     ):

#         case_ids = []

#         appearance_list = []

#         shape_list = []

#         boundary_list = []

#         key_list = []

#         for batch_idx, batch in enumerate(dataloader):
#             # if batch_idx == 2:
#             #   break


#             image = batch["image"].to(
#                 self.device
#             )

#             mask = batch["mask"].to(
#                 self.device
#             )

#             mask = mask.unsqueeze(1)

#             case_id = batch["case_id"][0]

#             backbone_output = self.backbone(image)
#             logits = backbone_output["logits"]
#             bottleneck = backbone_output["bottleneck"]

#             prototypes = (
#                 self.prototype_generator(
#                     bottleneck,
#                     mask
#                 )
#             )

#             case_ids.append(
#                 case_id
#             )

#             appearance_list.append(
#                 prototypes["appearance"].cpu()
#             )

#             shape_list.append(
#                 prototypes["shape"].cpu()
#             )

#             boundary_list.append(
#                 prototypes["boundary"].cpu()
#             )

#             key_list.append(
#                 prototypes["key"].cpu()
#             )

#             if (batch_idx + 1) % 10 == 0:

#                 print(
#                     f"Processed "
#                     f"{batch_idx + 1} cases"
#                 )

#         memory_bank = {

#             "case_ids": case_ids,

#             "appearance": torch.cat(
#                 appearance_list,
#                 dim=0
#             ),

#             "shape": torch.cat(
#                 shape_list,
#                 dim=0
#             ),

#             "boundary": torch.cat(
#                 boundary_list,
#                 dim=0
#             ),

#             "keys": torch.cat(
#                 key_list,
#                 dim=0
#             )
#         }

#         return memory_bank

In [40]:
# memory_loader = DataLoader(
#     memory_dataset,
#     batch_size=1,
#     shuffle=False,
#     num_workers=2
# )

In [41]:
# builder = MemoryBankBuilder(
#     backbone=backbone,
#     prototype_generator=prototype_generator,
#     device=device
# )

In [42]:
# memory_bank = builder.build_memory_bank(
#     memory_loader
# )

In [43]:
MEMORY_BANK_PATH = os.path.join(
    PROJECT_DIR,
    "memory_bank.pt"
)

memory_bank = torch.load(
    MEMORY_BANK_PATH,
    map_location="cpu"
)

In [44]:
# print(memory_bank["appearance"].shape)
# print(memory_bank["shape"].shape)
# print(memory_bank["boundary"].shape)

In [45]:
# print(memory_bank["appearance"].shape)

# print(memory_bank["shape"].shape)

# print(memory_bank["boundary"].shape)

# print(memory_bank["keys"].shape)

In [46]:
# print(memory_bank.keys())

# print(memory_bank["case_ids"][0])

# print(memory_bank["appearance"][0].shape)

# print(memory_bank["keys"][0].shape)

In [47]:
# torch.save(
#     memory_bank,
#     os.path.join(
#         PROJECT_DIR,
#         "memory_bank.pt"
#     )
# )

In [48]:
class AdaptiveGatingNetwork(nn.Module):

    def __init__(self):

        super().__init__()

        self.gate = nn.Sequential(

            nn.Linear(
                256 + 128 + 128,
                256
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                256,
                3
            )
        )

    def forward(
        self,
        appearance_proto,
        shape_proto,
        boundary_proto
    ):

        x = torch.cat(
            [
                appearance_proto,
                shape_proto,
                boundary_proto
            ],
            dim=1
        )

        weights = self.gate(x)

        weights = F.softmax(
            weights,
            dim=1
        )

        return weights

In [49]:
gating_network = AdaptiveGatingNetwork().to(device)

# appearance = torch.randn(
#     1,
#     256
# ).to(device)

# shape = torch.randn(
#     1,
#     128
# ).to(device)

# boundary = torch.randn(
#     1,
#     128
# ).to(device)

# weights = gating_network(
#     appearance,
#     shape,
#     boundary
# )

# print(weights)
# print(weights.shape)

# print(
#     weights.sum(dim=1)
# )

In [50]:
class MemoryRetriever(nn.Module):

    def __init__(
        self,
        memory_bank,
        gating_network,
        top_k=5
    ):

        super().__init__()

        self.memory_bank = memory_bank

        self.gating_network = gating_network

        self.top_k = top_k

    def forward(
        self,
        appearance_proto,
        shape_proto,
        boundary_proto
    ):

        appearance_bank = (
            self.memory_bank["appearance"]
            .to(appearance_proto.device)
        )

        shape_bank = (
            self.memory_bank["shape"]
            .to(shape_proto.device)
        )

        boundary_bank = (
            self.memory_bank["boundary"]
            .to(boundary_proto.device)
        )

        # ------------------
        # similarities
        # ------------------

        Sa = torch.matmul(
            appearance_proto,
            appearance_bank.T
        )

        Ss = torch.matmul(
            shape_proto,
            shape_bank.T
        )

        Sb = torch.matmul(
            boundary_proto,
            boundary_bank.T
        )

        # ------------------
        # adaptive weights
        # ------------------

        weights = self.gating_network(
            appearance_proto,
            shape_proto,
            boundary_proto
        )

        alpha = weights[:,0:1]

        beta = weights[:,1:2]

        gamma = weights[:,2:3]

        # ------------------
        # combined similarity
        # ------------------

        similarity = (
            alpha * Sa
            +
            beta * Ss
            +
            gamma * Sb
        )

        effective_k = min(
        self.top_k,
        similarity.shape[1]
        )

        scores, indices = torch.topk(
          similarity,
          k=effective_k,
          dim=1
        )

        return {

            "scores": scores,

            "indices": indices,

            "weights": weights,

            "Sa": Sa,

            "Ss": Ss,

            "Sb": Sb
        }

In [51]:
retriever = MemoryRetriever(
    memory_bank,
    gating_network,
    top_k=5
)
# appearance_proto = appearance_proto.to(device)
# shape_proto = shape_proto.to(device)
# boundary_proto = boundary_proto.to(device)
# outputs = retriever(
#     appearance_proto,
#     shape_proto,
#     boundary_proto
# )

In [52]:
# appearance = memory_bank[
#     "appearance"
# ][0:1]

# shape = memory_bank[
#     "shape"
# ][0:1]

# boundary = memory_bank[
#     "boundary"
# ][0:1]

In [53]:
# outputs = retriever(
#     appearance.to(device),
#     shape.to(device),
#     boundary.to(device)
# )

# print(
#     outputs["scores"].shape
# )


# print(
#     outputs["indices"].shape
# )

# print(
#     outputs["weights"]
# )

In [54]:
# print(outputs["scores"])
# print(outputs["indices"])

In [55]:
# print(outputs["Sa"])
# print(outputs["Ss"])
# print(outputs["Sb"])

In [56]:
class MemoryFeatureAggregator(nn.Module):

    def __init__(self, memory_bank):

        super().__init__()

        self.memory_bank = memory_bank

    def forward(
        self,
        scores,
        indices
    ):

        appearance_bank = (
            self.memory_bank["appearance"]
            .to(scores.device)
        )

        shape_bank = (
            self.memory_bank["shape"]
            .to(scores.device)
        )

        boundary_bank = (
            self.memory_bank["boundary"]
            .to(scores.device)
        )

        # --------------------
        # retrieval weights
        # --------------------

        weights = F.softmax(
            scores,
            dim=1
        )

        batch_size = indices.shape[0]

        appearance_features = []

        shape_features = []

        boundary_features = []

        for b in range(batch_size):

            idx = indices[b]

            w = weights[b].unsqueeze(1)

            retrieved_appearance = (
                appearance_bank[idx]
            )

            retrieved_shape = (
                shape_bank[idx]
            )

            retrieved_boundary = (
                boundary_bank[idx]
            )

            appearance_memory = (
                retrieved_appearance * w
            ).sum(dim=0)

            shape_memory = (
                retrieved_shape * w
            ).sum(dim=0)

            boundary_memory = (
                retrieved_boundary * w
            ).sum(dim=0)

            appearance_features.append(
                appearance_memory
            )

            shape_features.append(
                shape_memory
            )

            boundary_features.append(
                boundary_memory
            )

        appearance_features = torch.stack(
            appearance_features,
            dim=0
        )

        shape_features = torch.stack(
            shape_features,
            dim=0
        )

        boundary_features = torch.stack(
            boundary_features,
            dim=0
        )

        memory_feature = torch.cat(
            [
                appearance_features,
                shape_features,
                boundary_features
            ],
            dim=1
        )

        return {

            "appearance_memory":
                appearance_features,

            "shape_memory":
                shape_features,

            "boundary_memory":
                boundary_features,

            "memory_feature":
                memory_feature
        }

In [57]:
aggregator = MemoryFeatureAggregator(
    memory_bank
)

# agg_outputs = aggregator(
#     outputs["scores"],
#     outputs["indices"]
# )

In [58]:
# for k,v in agg_outputs.items():

#     print(
#         k,
#         v.shape
#     )

In [59]:
class CrossAttentionFusion(nn.Module):

    def __init__(
        self,
        feature_dim=768,
        memory_dim=512,
        num_heads=8
    ):

        super().__init__()

        self.memory_projection = nn.Linear(
            memory_dim,
            feature_dim
        )

        self.cross_attention = (
            nn.MultiheadAttention(
                embed_dim=feature_dim,
                num_heads=num_heads,
                batch_first=True
            )
        )

        self.norm = nn.LayerNorm(
            feature_dim
        )

    def forward(
        self,
        bottleneck,
        memory_feature
    ):

        B, C, D, H, W = (
            bottleneck.shape
        )

        bottleneck_tokens = (
            bottleneck
            .flatten(2)
            .transpose(1,2)
        )

        memory_token = (
            self.memory_projection(
                memory_feature
            )
            .unsqueeze(1)
        )

        attended, _ = (
            self.cross_attention(
                query=bottleneck_tokens,
                key=memory_token,
                value=memory_token
            )
        )

        fused_tokens = self.norm(
            bottleneck_tokens
            +
            attended
        )

        fused_bottleneck = (
            fused_tokens
            .transpose(1,2)
            .reshape(
                B,
                C,
                D,
                H,
                W
            )
        )

        return fused_bottleneck

In [60]:
fusion = CrossAttentionFusion().to(device)

# dummy_bottleneck = torch.randn(
#     1,
#     768,
#     3,
#     3,
#     3
# ).to(device)

# dummy_memory = torch.randn(
#     1,
#     512
# ).to(device)

# fused = fusion(
#     dummy_bottleneck,
#     dummy_memory
# )

# print(
#     fused.shape
# )

In [61]:
# for name, module in stage2_model.named_children():
#     print(name)

In [62]:
# x = torch.randn(
#     1,
#     4,
#     96,
#     96,
#     96
# ).to(device)

# with torch.no_grad():

#     hidden = stage2_model.swinViT(x)

# for i, h in enumerate(hidden):

#     print(
#         i,
#         h.shape
#     )

In [63]:
# print(stage2_model.encoder1)
# print()

# print(stage2_model.encoder2)
# print()

# print(stage2_model.encoder3)
# print()

# print(stage2_model.encoder4)
# print()

# print(stage2_model.encoder10)

In [64]:
# print(stage2_model.decoder5)
# print()

# print(stage2_model.decoder4)
# print()

# print(stage2_model.decoder3)
# print()

# print(stage2_model.decoder2)
# print()

# print(stage2_model.decoder1)

In [65]:
# import inspect

# print(
#     inspect.signature(
#         stage2_model.decoder5.forward
#     )
# )

In [66]:
# print(stage2_model.out)

In [67]:
# x = torch.randn(
#     1,
#     4,
#     96,
#     96,
#     96
# ).to(device)

# hidden = stage2_model.swinViT(x)

# enc0 = stage2_model.encoder1(x)
# #
# enc1 = stage2_model.encoder2(hidden[0])

# enc2 = stage2_model.encoder3(hidden[1])

# enc3 = stage2_model.encoder4(hidden[2])

# enc4 = stage2_model.encoder10(hidden[4])

# print(enc0.shape)
# print(enc1.shape)
# print(enc2.shape)
# print(enc3.shape)
# print(enc4.shape)

In [68]:
class Stage3Model(nn.Module):

    def __init__(
        self,
        stage2_model,
        prototype_generator,
        retriever,
        aggregator,
        fusion
    ):

        super().__init__()

        self.stage2_model = stage2_model

        self.prototype_generator = (
            prototype_generator
        )

        self.retriever = retriever

        self.aggregator = aggregator

        self.fusion = fusion

    def forward(self, x):

        # -------------------------
        # Encoder
        # -------------------------

        hidden = (
            self.stage2_model.swinViT(x)
        )

        enc0 = (
            self.stage2_model.encoder1(x)
        )

        enc1 = (
            self.stage2_model.encoder2(
                hidden[0]
            )
        )

        enc2 = (
            self.stage2_model.encoder3(
                hidden[1]
            )
        )

        enc3 = (
            self.stage2_model.encoder4(
                hidden[2]
            )
        )

        enc4 = (
            self.stage2_model.encoder10(
                hidden[4]
            )
        )

        # -------------------------
        # Initial Segmentation
        # -------------------------

        dec3 = (
            self.stage2_model.decoder5(
                enc4,
                hidden[3]
            )
        )

        dec2 = (
            self.stage2_model.decoder4(
                dec3,
                enc3
            )
        )

        dec1 = (
            self.stage2_model.decoder3(
                dec2,
                enc2
            )
        )

        dec0 = (
            self.stage2_model.decoder2(
                dec1,
                enc1
            )
        )

        out = (
            self.stage2_model.decoder1(
                dec0,
                enc0
            )
        )

        initial_logits = (
            self.stage2_model.out(out)
        )

        # -------------------------
        # Soft Mask
        # -------------------------

        mask_prob = torch.softmax(
            initial_logits,
            dim=1
        )[:, 1:2]

        # -------------------------
        # Prototype Generation
        # -------------------------

        prototypes = (
            self.prototype_generator(
                hidden[4],
                mask_prob
            )
        )

        # -------------------------
        # Retrieval
        # -------------------------

        retrieval = (
            self.retriever(
                prototypes["appearance"],
                prototypes["shape"],
                prototypes["boundary"]
            )
        )

        # -------------------------
        # Aggregation
        # -------------------------

        memory_outputs = (
            self.aggregator(
                retrieval["scores"],
                retrieval["indices"]
            )
        )

        # -------------------------
        # Fusion
        # -------------------------

        fused_enc4 = (
            self.fusion(
                enc4,
                memory_outputs[
                    "memory_feature"
                ]
            )
        )

        # -------------------------
        # Refined Decoder
        # -------------------------

        dec3_ref = (
            self.stage2_model.decoder5(
                fused_enc4,
                hidden[3]
            )
        )

        dec2_ref = (
            self.stage2_model.decoder4(
                dec3_ref,
                enc3
            )
        )

        dec1_ref = (
            self.stage2_model.decoder3(
                dec2_ref,
                enc2
            )
        )

        dec0_ref = (
            self.stage2_model.decoder2(
                dec1_ref,
                enc1
            )
        )

        out_ref = (
            self.stage2_model.decoder1(
                dec0_ref,
                enc0
            )
        )

        refined_logits = (
            self.stage2_model.out(
                out_ref
            )
        )

        return {

            "initial_logits":
                initial_logits,

            "refined_logits":
                refined_logits,

            "retrieval":
                retrieval,

            "memory":
                memory_outputs,

            "prototypes":
                prototypes
        }

In [69]:
stage3_model = Stage3Model(
    stage2_model=stage2_model,
    prototype_generator=prototype_generator,
    retriever=retriever,
    aggregator=aggregator,
    fusion=fusion
)
stage3_model = stage3_model.to(device)

In [70]:
# x = torch.randn(
#     1,
#     4,
#     96,
#     96,
#     96
# ).to(device)

# with torch.no_grad():

#     outputs = stage3_model(x)

In [71]:
# print(
#     outputs["initial_logits"].shape
# )

# print(
#     outputs["refined_logits"].shape
# )

# print(
#     outputs["prototypes"]["appearance"].shape
# )

# print(
#     outputs["prototypes"]["shape"].shape
# )

# print(
#     outputs["prototypes"]["boundary"].shape
# )

# print(
#     outputs["memory"]["memory_feature"].shape
# )

# print(
#     outputs["retrieval"]["scores"].shape
# )

# print(
#     outputs["retrieval"]["indices"].shape
# )

In [72]:
# outputs = stage3_model(x)

# print(
#     outputs["retrieval"]["weights"]
# )

# print(
#     outputs["retrieval"]["scores"]
# )

In [73]:
from monai.losses import DiceLoss

dice_loss = DiceLoss(
    sigmoid=False,
    softmax=True,
    to_onehot_y=True
)

ce_loss = nn.CrossEntropyLoss()


def compute_stage3_loss(
    outputs,
    target
):

    initial_logits = outputs[
        "initial_logits"
    ]

    refined_logits = outputs[
        "refined_logits"
    ]

    target_long = target.long()

    initial_dice = dice_loss(
        initial_logits,
        target_long.unsqueeze(1)
    )

    initial_ce = ce_loss(
        initial_logits,
        target_long
    )

    initial_loss = (
        initial_dice +
        initial_ce
    )

    refined_dice = dice_loss(
        refined_logits,
        target_long.unsqueeze(1)
    )

    refined_ce = ce_loss(
        refined_logits,
        target_long
    )

    refined_loss = (
        refined_dice +
        refined_ce
    )

    total_loss = (
        0.3 * initial_loss
        +
        0.7 * refined_loss
    )

    return {
        "total_loss": total_loss,
        "initial_loss": initial_loss,
        "refined_loss": refined_loss
    }

In [74]:
train_dataset = HNCDataset(
    METADATA_PATH,
    split="train"
)

val_dataset = HNCDataset(
    METADATA_PATH,
    split="val"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train Cases:", len(train_dataset))
print("Val Cases:", len(val_dataset))

Train Cases: 149
Val Cases: 32


In [75]:
# sample = next(
#     iter(train_loader)
# )

# print(
#     sample["image"].shape
# )

# print(
#     sample["mask"].shape
# )

In [76]:
# image = sample["image"].to(device)

# mask = sample["mask"].to(device)

# outputs = stage3_model(image)

# losses = compute_stage3_loss(
#     outputs,
#     mask
# )

# for k,v in losses.items():

#     print(
#         k,
#         v.item()
#     )

In [77]:
import torch

print(
    f"Allocated: "
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    f"Reserved: "
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

Allocated: 0.25 GB
Reserved: 0.27 GB


In [78]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [79]:
import gc
import torch

if 'backbone' in locals():
    del backbone
if 'builder' in locals():
    del builder
if 'memory_loader' in locals():
    del memory_loader

gc.collect()
torch.cuda.empty_cache()

In [80]:
import torch

print(
    f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    f"Reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

Allocated: 0.25 GB
Reserved: 0.27 GB


In [81]:
for param in stage3_model.stage2_model.parameters():
    param.requires_grad = False

In [82]:
for param in stage3_model.fusion.parameters():
    param.requires_grad = True

for param in stage3_model.prototype_generator.parameters():
    param.requires_grad = True

for param in stage3_model.retriever.gating_network.parameters():
    param.requires_grad = True

In [83]:

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        stage3_model.parameters()
    ),
    lr=1e-4,
    weight_decay=1e-5
)

scaler = torch.cuda.amp.GradScaler()

print("Optimizer created")

Optimizer created


/tmp/ipykernel_1749/152424703.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [84]:
def dice_score(
    prediction,
    target,
    eps=1e-8
):

    prediction = prediction.float()
    target = target.float()

    intersection = (
        prediction * target
    ).sum()

    union = (
        prediction.sum()
        +
        target.sum()
    )

    dice = (
        2.0 * intersection + eps
    ) / (
        union + eps
    )

    return dice.item()

In [85]:
def iou_score(
    prediction,
    target,
    eps=1e-8
):

    prediction = prediction.float()
    target = target.float()

    intersection = (
        prediction * target
    ).sum()

    union = (
        prediction
        +
        target
        -
        prediction * target
    ).sum()

    iou = (
        intersection + eps
    ) / (
        union + eps
    )

    return iou.item()

In [86]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0

    for batch in loader:

        image = batch["image"].to(device)

        mask = batch["mask"].to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():

            outputs = model(image)

            losses = compute_stage3_loss(
                outputs,
                mask
            )

            loss = losses["total_loss"]

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

    return running_loss / len(loader)

In [87]:
@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    device
):

    model.eval()

    total_loss = 0.0

    dice_list = []

    iou_list = []

    for batch in loader:

        image = batch["image"].to(device)

        mask = batch["mask"].to(device)

        with torch.cuda.amp.autocast():

          outputs = model(image)

          losses = compute_stage3_loss(
            outputs,
            mask
          )

        total_loss += (
            losses["total_loss"].item()
        )

        prediction = torch.argmax(
            outputs["refined_logits"],
            dim=1
        )

        dice = dice_score(
            prediction,
            mask
        )

        iou = iou_score(
            prediction,
            mask
        )

        dice_list.append(
            dice
        )

        iou_list.append(
            iou
        )

    avg_loss = (
        total_loss /
        len(loader)
    )

    avg_dice = (
        np.mean(dice_list)
    )

    avg_iou = (
        np.mean(iou_list)
    )

    return {
        "loss": avg_loss,
        "dice": avg_dice,
        "iou": avg_iou
    }

In [88]:
def save_checkpoint(
    model,
    optimizer,
    epoch,
    best_dice,
    path
):

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "best_dice":
                best_dice
        },
        path
    )

In [89]:
print(len(train_dataset))
print(len(val_dataset))
print(memory_bank["keys"].shape)
print(len(memory_bank["case_ids"]))

149
32
torch.Size([149, 512])
149


In [90]:
history = {
    "train_loss": [],
    "val_loss": [],
    "val_dice": [],
    "val_iou": []
}

In [91]:
stage3_model.train()

Stage3Model(
  (stage2_model): SwinUNETR(
    (swinViT): SwinTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv3d(4, 48, kernel_size=(2, 2, 2), stride=(2, 2, 2))
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (layers1): ModuleList(
        (0): BasicLayer(
          (blocks): ModuleList(
            (0-1): 2 x SwinTransformerBlock(
              (norm1): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
              (attn): WindowAttention(
                (qkv): Linear(in_features=48, out_features=144, bias=True)
                (attn_drop): Dropout(p=0.0, inplace=False)
                (proj): Linear(in_features=48, out_features=48, bias=True)
                (proj_drop): Dropout(p=0.0, inplace=False)
                (softmax): Softmax(dim=-1)
              )
              (drop_path): Identity()
              (norm2): LayerNorm((48,), eps=1e-05, elementwise_affine=True)
              (mlp): MLPBlock(
                (linear1): Linear(in_features=48

In [92]:
NUM_EPOCHS = 30

BEST_STAGE3_MODEL = os.path.join(
    CHECKPOINT_DIR,
    "best_stage3_model.pth"
)

best_dice = 0.0

for epoch in range(NUM_EPOCHS):

    train_loss = train_one_epoch(
        stage3_model,
        train_loader,
        optimizer,
        device
    )

    val_metrics = validate_one_epoch(
        stage3_model,
        val_loader,
        device
    )
    import gc

    gc.collect()
    torch.cuda.empty_cache()
    history["train_loss"].append(train_loss)

    history["val_loss"].append(
      val_metrics["loss"]
    )

    history["val_dice"].append(
      val_metrics["dice"]
    )

    history["val_iou"].append(
      val_metrics["iou"]
    )

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS}"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Val Loss: {val_metrics['loss']:.4f}"
    )

    print(
        f"Val Dice: {val_metrics['dice']:.4f}"
    )

    print(
        f"Val IoU: {val_metrics['iou']:.4f}"
    )

    print(
        f"Best Dice So Far: {best_dice:.4f}"
    )

    if val_metrics["dice"] > best_dice:

        best_dice = val_metrics["dice"]

        save_checkpoint(
            stage3_model,
            optimizer,
            epoch,
            best_dice,
            BEST_STAGE3_MODEL
        )

        print(
            "Best model saved"
        )

/tmp/ipykernel_1749/943891052.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_1749/2295962569.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/30
Train Loss: 0.1612
Val Loss: 0.2627
Val Dice: 0.6602
Val IoU: 0.5160
Best Dice So Far: 0.0000
Best model saved
Epoch 2/30
Train Loss: 0.1606
Val Loss: 0.2646
Val Dice: 0.6526
Val IoU: 0.5056
Best Dice So Far: 0.6602
Epoch 3/30
Train Loss: 0.1646
Val Loss: 0.2380
Val Dice: 0.6867
Val IoU: 0.5395
Best Dice So Far: 0.6602
Best model saved
Epoch 4/30
Train Loss: 0.1614
Val Loss: 0.2679
Val Dice: 0.6578
Val IoU: 0.5206
Best Dice So Far: 0.6867
Epoch 5/30
Train Loss: 0.1644
Val Loss: 0.2518
Val Dice: 0.6817
Val IoU: 0.5400
Best Dice So Far: 0.6867
Epoch 6/30
Train Loss: 0.1594
Val Loss: 0.2395
Val Dice: 0.6815
Val IoU: 0.5387
Best Dice So Far: 0.6867
Epoch 7/30
Train Loss: 0.1679
Val Loss: 0.2610
Val Dice: 0.6665
Val IoU: 0.5242
Best Dice So Far: 0.6867
Epoch 8/30
Train Loss: 0.1623
Val Loss: 0.2286
Val Dice: 0.7016
Val IoU: 0.5624
Best Dice So Far: 0.6867
Best model saved
Epoch 9/30
Train Loss: 0.1598
Val Loss: 0.2584
Val Dice: 0.6825
Val IoU: 0.5403
Best Dice So Far: 0.7016
Epoc

In [93]:
torch.save(
    history,
    os.path.join(
        CHECKPOINT_DIR,
        "stage3_training_history.pt"
    )
)

In [96]:
CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "best_stage3_model.pth"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

stage3_model.load_state_dict(
    checkpoint["model_state_dict"]
)

stage3_model.eval()

print("Best Stage3 model loaded")

Best Stage3 model loaded


In [99]:
test_dataset = HNCDataset(
    METADATA_PATH,
    split="test"
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Test Cases:", len(test_dataset))

Test Cases: 32


In [100]:
test_metrics = validate_one_epoch(
    model=stage3_model,
    loader=test_loader,
    device=device
)

print(
    f"Test Loss: {test_metrics['loss']:.4f}"
)

print(
    f"Test Dice: {test_metrics['dice']:.4f}"
)

print(
    f"Test IoU: {test_metrics['iou']:.4f}"
)

/tmp/ipykernel_1749/2295962569.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Test Loss: 0.1899
Test Dice: 0.6936
Test IoU: 0.5448
